In [ ]:
# Install and download a book
!wget https://www.gutenberg.org/files/11/11-0.txt -O alice.txt

In [ ]:
# Load the text

with open("alice.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Characters in book:", len(text))
print(text[:500])

In [ ]:
# Converts text into raw bytes
data = list(text.encode("utf-8"))
print("First 20 bytes:", data[:20])

In [4]:
# Helper functions for BPE
from collections import Counter

def get_pair_freq(tokens):
    pairs = Counter()
    for i in range(len(tokens)-1):
        pairs[(tokens[i], tokens[i+1])] += 1
    return pairs


def merge_pair(tokens, pair, new_id):
    i = 0
    new_tokens = []

    while i < len(tokens):
        if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == pair:
            new_tokens.append(new_id)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return new_tokens

In [5]:
# train BPE

def train_bpe(data, vocab_size):

    tokens = data[:]
    merges = {}

    current_id = 256   # after byte tokens

    while current_id < vocab_size:

        pair_freq = get_pair_freq(tokens)
        if not pair_freq:
            break

        best_pair = max(pair_freq, key=pair_freq.get)

        tokens = merge_pair(tokens, best_pair, current_id)

        merges[best_pair] = current_id
        current_id += 1

    return tokens, merges

In [6]:
# Measure compression ratio

def compression_ratio(original, compressed):
    return len(original) / len(compressed)

In [ ]:
# Experiment with different vocabulary size

vocab_sizes = [256, 300, 400, 600, 1000]

results = []

for v in vocab_sizes:

    tokens, merges = train_bpe(data, v)

    ratio = compression_ratio(data, tokens)

    results.append((v, len(tokens), ratio))

for r in results:
    print(f"Vocab: {r[0]} | Tokens after BPE: {r[1]} | Compression Ratio: {r[2]:.2f}")

In [ ]:
# Plot the effect

import matplotlib.pyplot as plt

vocab = [r[0] for r in results]
ratio = [r[2] for r in results]

plt.plot(vocab, ratio, marker='o')
plt.xlabel("Vocabulary Size")
plt.ylabel("Compression Ratio")
plt.title("Effect of Vocabulary Size on BPE Compression")
plt.show()

In [ ]:
''' 1. Small vocabulary → fewer merges → words split into many tokens → low compression.

2. Larger vocabulary → more merges → common letter pairs become single tokens.

3. This reduces the number of tokens, so the text becomes more compressed.

4. But very large vocabulary increases model memory and embedding size, so models choose a balanced size (like in GPT-2).'''